Step 1: Install PyMuPDF
This library allows Python to read PDF documents and extract their text.
You only need to install it once in your Python environment.

In [1]:
%pip install pymupdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Step 2: Import the library

In [2]:
import pymupdf

print("PyMuPDF imported successfully!")

PyMuPDF imported successfully!


Step 3: Open the PDF and count its pages

In [3]:
from pathlib import Path

# Find the PDF in our data folder
pdf_path = next(Path("data").glob("*.pdf"))

# Open the PDF
pdf = pymupdf.open(pdf_path)

# Display the filename and number of pages
print("PDF:", pdf_path.name)
print("Number of pages:", len(pdf))

PDF: Module 1_ The Role of a Volunteer Coach.pdf
Number of pages: 4


Path("data") — points to your data folder.
.glob("*.pdf") — finds PDF files in that folder.
next() — selects the first matching PDF.
pymupdf.open() — opens the PDF.
len(pdf) — counts its pages.

In [4]:
# Get the first page (Python starts counting at 0)
page = pdf[0]

# Extract its text
text = page.get_text()

# Display the first 500 characters
print(text[:500])

Module 1: The Role of a Volunteer 
Coach 
Learning Objectives 
This module introduces the core responsibilities, mindset, and behaviours that define 
effective coaching. It highlights how coaching differs from teaching or mentoring and 
provides foundational practices for supporting learners. 
 
Skills and Competencies 
Coaching vs. Teaching vs. Mentoring 
●​ Teaching: Delivers knowledge and skills. 
●​ Mentoring: Offers advice based on experience. 
●​ Coaching: Helps learners discover solutions


what's happening above
pdf[0] — selects the first page.
page.get_text() — extracts the text from that page.
text[:500] — displays only the first 500 characters, so we don't overwhelm the output.

To extract text from all four pages, we need a for loop.

In [5]:
# Create an empty list to store the text
pages_text = []

# Loop through every page in the PDF
for page in pdf:
    text = page.get_text()
    pages_text.append(text)

# Check our results
print(f"Extracted text from {len(pages_text)} pages")

Extracted text from 4 pages


What's happening?
1. pages_text = [] creates an empty list.
2. for page in pdf: loops through all four pages.
3. page.get_text() extracts the text from the current page.
4. .append(text) adds that page's text to our list.

In [6]:
#This displays the first 200 characters of page 3.
print(pages_text[2][:200])

learner feel more prepared and reduce uncertainty. For example, you might say, “By the 
end of this session, it would be great if you feel more confident with "xyz". 
My role is to help you think thro


Step 6: Organise our extracted text
Before chunking, let's organise the text so we know which page each piece of text came from.

Remember your first RAG project, where you created a list of dictionaries?

We're going to do something similar, but this time Python will create the dictionaries for us.

In [7]:
documents = []

for page_number, text in enumerate(pages_text, start=1):
    documents.append({
        "page": page_number,
        "text": text
    })

print(f"Created {len(documents)} documents")

Created 4 documents


what is happeinhg 
documents = [] creates an empty list.
enumerate(pages_text, start=1) gives each page a number, starting at 1 rather than 0.
page_number contains the page number.
text contains the extracted text from that page.
.append() adds a new dictionary to the list.

In [8]:
#inspect the first dictionary in the list
print(documents[0])

{'page': 1, 'text': 'Module 1: The Role of a Volunteer \nCoach \nLearning Objectives \nThis module introduces the core responsibilities, mindset, and behaviours that define \neffective coaching. It highlights how coaching differs from teaching or mentoring and \nprovides foundational practices for supporting learners. \n \nSkills and Competencies \nCoaching vs. Teaching vs. Mentoring \n●\u200b Teaching: Delivers knowledge and skills. \n●\u200b Mentoring: Offers advice based on experience. \n●\u200b Coaching: Helps learners discover solutions through inquiry and reflection. \nCoaching focuses on empowering learners to think independently and develop \nconfidence. \nSetting Expectations \n●\u200b Communicate goals, roles, and responsibilities clearly. \n●\u200b Establish realistic expectations for both coach and learner. \n●\u200b Align on what success looks like during a session. \nBuilding Trust and Psychological Safety \n●\u200b Create an environment where learners feel safe to ask qu

When we later split the text into chunks, we can preserve the page number. That will allow our RAG application to say something like:

Answer: A volunteer coach helps learners discover solutions.

Source: Page 1.

That's called source attribution, and it's a valuable feature in RAG applications.

1. Create our chunking function
Remember, chunking means splitting longer text into smaller, manageable pieces.

We'll aim for approximately 60 words per chunk, keeping sentences together where possible.

In [9]:
import re

def chunk_text(text, max_words=60):
    """Split text into chunks without cutting sentences."""

    # Split the text into sentences
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    chunks = []
    current = []
    word_count = 0

    for sentence in sentences:
        n_words = len(sentence.split())

        # Check whether adding this sentence exceeds our limit
        if current and word_count + n_words > max_words:
            chunks.append(" ".join(current))

            current = []
            word_count = 0

        # Add the sentence to the current chunk
        current.append(sentence)
        word_count += n_words

    # Save the final chunk
    if current:
        chunks.append(" ".join(current))

    return chunks

This uses several concepts you've already learned:
- re.split() — splits the text into sentences using punctuation.
- chunks = [] — stores completed chunks.
- current = [] — collects sentences for the chunk we're building.
- word_count — counts words in the current chunk.
- .append() — adds a completed chunk to our list.
- return chunks — returns a list of strings.

In [10]:
all_chunks = []

for document in documents:
    page_number = document["page"]
    text = document["text"]

    for index, chunk in enumerate(chunk_text(text)):
        all_chunks.append({
            "page": page_number,
            "chunk_index": index,
            "text": chunk
        })

print(f"Created {len(all_chunks)} chunks")

Created 13 chunks


What is happening?
You've encountered nested for loops before!

The outer loop goes through each PDF page.

The inner loop breaks that page's text into chunks.

And enumerate() numbers each chunk, starting at zero for every new page.

Each chunk becomes a dictionary containing:

In [11]:
#inspect the chunks
for chunk in all_chunks:
    print(
        f"Page {chunk['page']} | "
        f"Chunk {chunk['chunk_index']} | "
        f"{len(chunk['text'].split())} words"
    )

Page 1 | Chunk 0 | 60 words
Page 1 | Chunk 1 | 49 words
Page 1 | Chunk 2 | 34 words
Page 2 | Chunk 0 | 49 words
Page 2 | Chunk 1 | 57 words
Page 2 | Chunk 2 | 53 words
Page 3 | Chunk 0 | 29 words
Page 3 | Chunk 1 | 53 words
Page 3 | Chunk 2 | 33 words
Page 3 | Chunk 3 | 113 words
Page 4 | Chunk 0 | 49 words
Page 4 | Chunk 1 | 47 words
Page 4 | Chunk 2 | 33 words


Page 3, Chunk 3 contains 113 words!
Remember, we set:
def chunk_text(text, max_words=60):
So why did we get 113 words?

Because our function tries to keep sentences together rather than cutting them in half.

If the extracted text contains a sentence or unbroken block of 113 words, our function keeps it together. Not an error but worth investigating 

In [12]:
#inspect the text of the first chunk
print(all_chunks[0]['text'])

Module 1: The Role of a Volunteer 
Coach 
Learning Objectives 
This module introduces the core responsibilities, mindset, and behaviours that define 
effective coaching. It highlights how coaching differs from teaching or mentoring and 
provides foundational practices for supporting learners. Skills and Competencies 
Coaching vs. Teaching vs. Mentoring 
●​ Teaching: Delivers knowledge and skills. ●​ Mentoring: Offers advice based on experience.


In [13]:
#inspect the second chunck
print(all_chunks[1]['text'])

●​ Coaching: Helps learners discover solutions through inquiry and reflection. Coaching focuses on empowering learners to think independently and develop 
confidence. Setting Expectations 
●​ Communicate goals, roles, and responsibilities clearly. ●​ Establish realistic expectations for both coach and learner. ●​ Align on what success looks like during a session.


Notice how you're combining list indexing and dictionary keys!

all_chunks[0] selects the first dictionary, and ["text"] selects its text.

In [14]:
for chunk in all_chunks:
    if chunk["page"] == 3 and chunk["chunk_index"] == 3:
        print(chunk["text"])

How to Foster Safety 
Demonstrate respect and active listening.​
Active listening is more than just hearing words, it shows the learner that you are fully 
present and genuinely interested in their thought process.​
Examples include:​
Reflecting back: 
“So it sounds like you’re stuck on how the function returns its value—did I get that 
right?”​
Pausing before responding: 
Giving the learner space to finish their thought instead of jumping in.​
Encouraging with prompts: 
“Tell me more about what you were thinking here.”​
Checking understanding: 
“When you say the code ‘doesn’t work,’ what behaviour are you expecting?”​
Non-verbal cues: 
●​ Nodding, maintaining open posture, and keeping your attention on the learner 
rather than your screen.​


Step 8: Store our PDF chunks in SQLite.
it's the same as the first RAG project, the difference us that we are now storing page numbers, so our RAG can identify which pdf page an answer came from

Step 8.1: Create the SQLite database
What's happening?
- import sqlite3 loads Python's built-in SQLite library.
- sqlite3.connect("rag_pdf.db") opens the database, or creates it if it doesn't exist.
- conn stores our database connection.
When you run this, a new file called rag_pdf.db should appear in your VS Code project folder.

In [15]:
import sqlite3

# Create or connect to our SQLite database
conn = sqlite3.connect("rag_pdf.db")

print("Database connected successfully!")

Database connected successfully!


Step 8.2: Create our chunks table

In [16]:
conn.execute("""
CREATE TABLE IF NOT EXISTS chunks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    page INTEGER,
    chunk_index INTEGER,
    text TEXT,
    embedding BLOB
)
""")

conn.commit()

print("Chunks table created!")

Chunks table created!


Step 8.3: Insert the chunks into SQLite

In [17]:
# Insert each chunk into the database
#Each ? is a placeholder for a value from the dictionary
#the triple quotes allow us to write a multi-line string for the SQL command. not a docstring, but a multi-line string for the SQL command.

for chunk in all_chunks:
    conn.execute(
        """
        INSERT INTO chunks (page, chunk_index, text)
        VALUES (?, ?, ?)
        """,
        (
            chunk["page"],
            chunk["chunk_index"],
            chunk["text"]
        )
    )

# Save our changes. note This commits/saves the database changes, not a Git commit!
conn.commit()

print(f"Inserted {len(all_chunks)} chunks into SQLite")

Inserted 13 chunks into SQLite


Step 8.4: Check our database

In [18]:
# Count how many chunks are stored

count = conn.execute(
    "SELECT COUNT(*) FROM chunks"
).fetchone()[0]

print(f"Database contains {count} chunks")

Database contains 26 chunks


one more check. Retrieve the page numbers and 

In [ ]:
# IMPORTANT:
# Run this cell only once when populating the database.
# Running it again will insert duplicate chunks.
rows = conn.execute(
    "SELECT id, page, chunk_index FROM chunks"
).fetchall()

for row in rows:
    print(row)

(1, 1, 0)
(2, 1, 1)
(3, 1, 2)
(4, 2, 0)
(5, 2, 1)
(6, 2, 2)
(7, 3, 0)
(8, 3, 1)
(9, 3, 2)
(10, 3, 3)
(11, 4, 0)
(12, 4, 1)
(13, 4, 2)
(14, 1, 0)
(15, 1, 1)
(16, 1, 2)
(17, 2, 0)
(18, 2, 1)
(19, 2, 2)
(20, 3, 0)
(21, 3, 1)
(22, 3, 2)
(23, 3, 3)
(24, 4, 0)
(25, 4, 1)
(26, 4, 2)


In [44]:
count = conn.execute(
    "SELECT COUNT(*) FROM chunks"
).fetchone()[0]

print(f"Total chunks: {count}")

Total chunks: 26


In [45]:
conn.execute("DELETE FROM chunks WHERE id > 13")
conn.commit()

print("Duplicate chunks deleted!")

Duplicate chunks deleted!


In [46]:
rows = conn.execute(
    "SELECT id, page, chunk_index FROM chunks ORDER BY id"
).fetchall()

for row in rows:
    print(row)

(1, 1, 0)
(2, 1, 1)
(3, 1, 2)
(4, 2, 0)
(5, 2, 1)
(6, 2, 2)
(7, 3, 0)
(8, 3, 1)
(9, 3, 2)
(10, 3, 3)
(11, 4, 0)
(12, 4, 1)
(13, 4, 2)


Meet the model: all-MiniLM-L6-v2
We're using all-MiniLM-L6-v2 — a small, pretrained, open-weight embedding model available on Hugging Face.
Feature	Description
Model	all-MiniLM-L6-v2
Library	SentenceTransformers
Model type	Text embedding model
Parameters	Approximately 22 million
Output	384 numbers per text embedding
Cost	Free to run locally
Training	Pretrained and fine-tuned for semantic similarity

What does it actually do?
It converts text into numerical representations called embeddings.

The model has learned to produce vectors that tend to be closer together when the texts have similar meanings.

That's what allows our RAG system to recognise that "What does a volunteer coach do?" is related to text about coaching responsibilities, even if the wording differs.

Is this training or inference?
Inference. We're using an already-trained model.

We're not changing its weights or training it on your volunteer coaching PDF.

And the model doesn't memorise your PDF. It simply converts each chunk into a vector that our system can search later.

a vector is also called an embedding 

Step 9: Generate embeddings for our 13 PDF chunks.

Step 9.1: Load our embedding model

In [20]:
from sentence_transformers import SentenceTransformer

# Load our pretrained embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

c:\Users\charl\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding model loaded!


In [21]:
# Extract the text from each chunk. This line is a list comprehension:
chunk_texts = [chunk["text"] for chunk in all_chunks]

# Generate embeddings. vectors = embedder.encode(chunk_texts, normalize_embeddings=True)The model converts each chunk into a 384-number vector.normalize_embeddings=True gives each vector a length of 1, which helps us calculate cosine similarity late
vectors = embedder.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", vectors.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]

Embedding matrix shape: (13, 384)


Step 9.3: Save the embeddings in SQLite

What is happening? 
zip() pairs each chunk ID with its corresponding embedding.
astype(np.float32) ensures each numerical value is stored as a 32-bit floating-point number.
.tobytes() converts the NumPy array into bytes for SQLite's BLOB column.
UPDATE modifies an existing database row rather than creating a new one.
conn.commit() saves the database changes.

In [22]:
import numpy as np

# Retrieve chunk IDs in a consistent order
rows = conn.execute(
    "SELECT id FROM chunks ORDER BY id"
).fetchall()

chunk_ids = [row[0] for row in rows]

# Store each embedding in the matching database row
for chunk_id, vector in zip(chunk_ids, vectors):
    vector_bytes = vector.astype(np.float32).tobytes()

    conn.execute(
        "UPDATE chunks SET embedding = ? WHERE id = ?",
        (vector_bytes, chunk_id)
    )

conn.commit()

print(f"Stored {len(vectors)} embeddings in SQLite")

Stored 13 embeddings in SQLite


In [23]:
count = conn.execute(
    "SELECT COUNT(*) FROM chunks WHERE embedding IS NOT NULL"
).fetchone()[0]

print(f"Chunks with embeddings: {count}")

Chunks with embeddings: 13


Your database now contains the PDF chunks, page numbers and embeddings.

Step 10.1: Load our embeddings from SQLite
Remember, your 13 embeddings are stored in SQLite as bytes (BLOB). We need to convert them back into NumPy arrays before comparing them.

In [24]:
import numpy as np

# Retrieve our chunks and their embeddings
rows = conn.execute("""
    SELECT id, page, chunk_index, text, embedding
    FROM chunks
    WHERE embedding IS NOT NULL
    ORDER BY id
""").fetchall()

# Create two empty lists
meta = []
vectors = []

for row in rows:
    chunk_id, page, chunk_index, text, embedding_bytes = row

    # Store the readable information
    meta.append({
        "chunk_id": chunk_id,
        "page": page,
        "chunk_index": chunk_index,
        "text": text
    })

    # Convert bytes back into a NumPy array
    vector = np.frombuffer(embedding_bytes, dtype=np.float32)
    vectors.append(vector)

# Stack the vectors into a 2D NumPy array
matrix = np.vstack(vectors)

print("Embedding matrix shape:", matrix.shape)
print("Metadata records:", len(meta))

Embedding matrix shape: (13, 384)
Metadata records: 13


Step 10.2: Create our retrieval function
Now we'll write the function that finds the most relevant chunks.

In [25]:
def retrieve(question, k=3):
    """Retrieve the k most relevant PDF chunks."""

    # Convert the question into an embedding
    question_vector = embedder.encode(
        [question],
        normalize_embeddings=True
    )[0]

    # Compare the question with all stored vectors. Calculate cosine similarity. remember, stored embeddings and question vector are normalised.
    scores = np.dot(matrix, question_vector)

    # Sort by similarity, highest first
    top_positions = np.argsort(scores)[::-1][:k]

    # Collect our results
    results = []

    for position in top_positions:
        chunk = meta[position]

        results.append({
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "text": chunk["text"],
            "score": float(scores[position])
        })

    return results

question_vector = embedder.encode(
    [question],
    normalize_embeddings=True
)[0]
Just as we embedded the PDF chunks, we now embed the user's question.

It produces another 384-number vector.
This combines three operations:
- np.argsort(scores) — returns indexes sorted from lowest score to highest.
- [::-1] — reverses the order.
- [:k] — keeps the first k results.
Since k=3, we retrieve the three highest-scoring chunks.

Step 10.3: Test your retrieval function

In [26]:
question = "What is the role of a volunteer coach?"

results = retrieve(question, k=3)

for result in results:
    print(
        f"Score: {result['score']:.3f} | "
        f"Page: {result['page']}"
    )
    print(result["text"])
    print()

Score: 0.678 | Page: 1
Module 1: The Role of a Volunteer 
Coach 
Learning Objectives 
This module introduces the core responsibilities, mindset, and behaviours that define 
effective coaching. It highlights how coaching differs from teaching or mentoring and 
provides foundational practices for supporting learners. Skills and Competencies 
Coaching vs. Teaching vs. Mentoring 
●​ Teaching: Delivers knowledge and skills. ●​ Mentoring: Offers advice based on experience.

Score: 0.553 | Page: 1
●​ Coaching: Helps learners discover solutions through inquiry and reflection. Coaching focuses on empowering learners to think independently and develop 
confidence. Setting Expectations 
●​ Communicate goals, roles, and responsibilities clearly. ●​ Establish realistic expectations for both coach and learner. ●​ Align on what success looks like during a session.

Score: 0.519 | Page: 2
Unlike teaching or mentoring, coaching emphasizes reflective questioning, 
active listening, and learner ownership

A cosine similarity score closer to 1 generally means the text is more similar in meaning to your question.
A score of 0.678 doesn't mean the answer is 67.8% correct.

It tells us how closely the question's embedding matches a particular chunk's embedding.

Step 11.1: Load our generator model
We've retrieved relevant chunks from your four-page volunteer coaching PDF. Now we'll use Flan-T5 to turn that retrieved information into a written answer.

In [27]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

print("Generator loaded!")

Device set to use cpu


Generator loaded!


In [28]:
def generate(prompt, max_new_tokens=100):
    """Generate an answer using Flan-T5."""

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    return output[0]["generated_text"]

In [47]:
def generate_beam(prompt, max_new_tokens=100):
    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=4
    )
    return output[0]["generated_text"]

In [29]:
def build_prompt(question, chunks):
    """Combine retrieved PDF chunks with the user's question."""

    chunk_texts = []

    for chunk in chunks:
        chunk_texts.append(
            f"Page {chunk['page']}: {chunk['text']}"
        )

    context = "\n\n".join(chunk_texts)

    return f"""Answer the question using only the context below.
If the answer is not in the context, say "I don't know".
Be concise and accurate.

Context:
{context}

Question: {question}
Answer:"""

What's happening?
You should recognise nearly everything here.
- chunk_texts = [] creates an empty list.
- The for loop extracts the text from each retrieved chunk.
- The f-string adds its PDF page number.
- "\n\n".join() combines the chunks into one block of text, separated by blank lines.
- The triple-quoted f-string creates the complete prompt.
The instructions tell Flan-T5 to base its answer on the retrieved context rather than guess.

In [30]:
def ask(question, k=3):
    """Retrieve relevant PDF chunks and generate an answer."""

    # 1. Retrieve the most relevant chunks
    chunks = retrieve(question, k=k)

    # 2. Build the prompt using those chunks
    prompt = build_prompt(question, chunks)

    # 3. Generate an answer
    answer = generate(prompt)

    # 4. Display the result
    print(f"Question: {question}")
    print(f"Answer: {answer}")

    print("\nSources:")

    for chunk in chunks:
        print(
            f"Page {chunk['page']} | "
            f"Similarity: {chunk['score']:.3f}"
        )

Notice that ask() calls three functions you created:

retrieve() → build_prompt() → generate()

Each function has one job. This makes the code easier to understand, test and maintain.

Step 11.5: Test your PDF RAG application

In [31]:
ask("What is the role of a volunteer coach?")

Question: What is the role of a volunteer coach?
Answer: Helps learners discover solutions through inquiry and reflection. Coaching focuses on empowering learners to think independently and develop confidence. Setting Expectations  Communicate goals, roles, and responsibilities clearly.  Establish realistic expectations for both coach and learner.  Align on what success looks like during a session.

Sources:
Page 1 | Similarity: 0.678
Page 1 | Similarity: 0.553
Page 2 | Similarity: 0.519


Step 12 — Evaluate your RAG system
what we've done so far:

complerted the entire RAG process, including PDF extraction, chunking, SQLite, embeddings, retrieval and generation.

Step 12.1: Test three types of questions
We'll test your PDF RAG application using:
1. Direct questions — information clearly stated in the PDF.
2. Paraphrased questions — the same idea, expressed differently.
3. Unanswerable questions — information not contained in the PDF.

In [32]:
# Test 1: Direct question
ask("What is the role of a volunteer coach?")

# Test 2: Paraphrased question
ask("Should coaches give learners the answers?")

# Test 3: Unanswerable question
ask("How much are volunteer coaches paid?")

Question: What is the role of a volunteer coach?
Answer: Helps learners discover solutions through inquiry and reflection. Coaching focuses on empowering learners to think independently and develop confidence. Setting Expectations  Communicate goals, roles, and responsibilities clearly.  Establish realistic expectations for both coach and learner.  Align on what success looks like during a session.

Sources:
Page 1 | Similarity: 0.678
Page 1 | Similarity: 0.553
Page 2 | Similarity: 0.519
Question: Should coaches give learners the answers?
Answer: Yes

Sources:
Page 1 | Similarity: 0.623
Page 2 | Similarity: 0.587
Page 2 | Similarity: 0.536
Question: How much are volunteer coaches paid?
Answer: I don't know

Sources:
Page 1 | Similarity: 0.464
Page 1 | Similarity: 0.320
Page 2 | Similarity: 0.318


note the answer to test 2 is probably wrong 
Your PDF describes coaching as helping learners discover solutions through inquiry and reflection, rather than simply giving them answers.
the results from test 2 suggest the retrieval text is symantically related to the question 

similarity scores don't tell us whether the chunks contain the correct answer.

In [33]:
question = "Should coaches give learners the answers?"

results = retrieve(question, k=3)

for result in results:
    print(f"Page: {result['page']}")
    print(f"Score: {result['score']:.3f}")
    print(result["text"])
    print("-" * 50)

Page: 1
Score: 0.623
●​ Coaching: Helps learners discover solutions through inquiry and reflection. Coaching focuses on empowering learners to think independently and develop 
confidence. Setting Expectations 
●​ Communicate goals, roles, and responsibilities clearly. ●​ Establish realistic expectations for both coach and learner. ●​ Align on what success looks like during a session.
--------------------------------------------------
Page: 2
Score: 0.587
Unlike teaching or mentoring, coaching emphasizes reflective questioning, 
active listening, and learner ownership. Why Coaching Matters 
Coaching strengthens problem-solving abilities and builds confidence, especially for 
beginners from underrepresented groups in tech. It supports independence and long-
term growth. Setting Expectations with Your 
Learner 
Begin each session by clarifying: 
●​ The Goal: What the learner wants to achieve.
--------------------------------------------------
Page: 2
Score: 0.536
Growth Mindset 
●​ Encour

In [48]:
question = "Should coaches give learners the answers?"

chunks = retrieve(question, k=3)
prompt = build_prompt(question, chunks)

print("Original:", generate(prompt))
print("Beam search:", generate_beam(prompt))

Original: Yes
Beam search: Yes


In [34]:
ask("According to the document, should a coach provide solutions directly or guide learners to discover their own solutions?")

Question: According to the document, should a coach provide solutions directly or guide learners to discover their own solutions?
Answer: guide learners to discover their own solutions

Sources:
Page 1 | Similarity: 0.629
Page 2 | Similarity: 0.582
Page 2 | Similarity: 0.558


You've already demonstrated that the embedding model retrieves relevant information. The weakness appears to be in how Flan-T5 interprets that information and generates an answer.

But there's a limit to what prompt engineering can achieve with a relatively small language model.

I'd approach this as an AI engineering experiment: improve what we have, measure the results, then decide whether a stronger model is necessary.

What could we improve?
Improvement	Why it might help- Priority
Prompt engineering	Give Flan-T5 clearer instructions- First
Reduce retrieved chunks	Remove irrelevant context and reduce truncation- Second
Improve chunking	Preserve more complete ideas and context- Third
Change generator model	Improve comprehension and answer generation- Later
Automated evaluation	Measure whether changes actually help- Essential

I suggest keeping your existing system as our baseline so we can compare improvements.

Improve our RAG generator
one controlled experiment: prompt engineering.

Keep your original build_prompt() function unchanged, so you can compare the results.

13.1 Create an improved prompt

In [35]:
def build_prompt_v2(question, chunks):
    """Build a more focused prompt for Flan-T5."""

    context = "\n\n".join(
        chunk["text"] for chunk in chunks
    )

    return f"""Read the context and answer the question.

Rules:
- Use only information from the context.
- Do not guess.
- If the answer is missing, say "I don't know".
- For yes/no questions, explain why.
- Keep the answer short.

Context:
{context}

Question: {question}
Answer:"""

13.2 Test the original failed question

In [36]:
question = "Should coaches give learners the answers?"

chunks = retrieve(question, k=3)

prompt = build_prompt_v2(question, chunks)

answer = generate(prompt)

print("Question:", question)
print("Answer:", answer)

Question: Should coaches give learners the answers?
Answer: For yes/no questions, explain why.


In [37]:
def build_prompt_v3(question, chunks):
    """Build a simple context-based prompt."""

    context = "\n\n".join(
        chunk["text"] for chunk in chunks
    )

    return f"""Context:
{context}

Based on the context, answer this question:
{question}

Answer:"""

In [38]:
question = "Should coaches give learners the answers?"

chunks = retrieve(question, k=1)

prompt = build_prompt_v3(question, chunks)

print("Answer:", generate(prompt))

Answer: (ii).


In [39]:
print(prompt)
print("\nPrompt length:", len(prompt.split()), "words")

Context:
●​ Coaching: Helps learners discover solutions through inquiry and reflection. Coaching focuses on empowering learners to think independently and develop 
confidence. Setting Expectations 
●​ Communicate goals, roles, and responsibilities clearly. ●​ Establish realistic expectations for both coach and learner. ●​ Align on what success looks like during a session.

Based on the context, answer this question:
Should coaches give learners the answers?

Answer:

Prompt length: 64 words


In [40]:
question = "How does coaching help learners?"

chunks = retrieve(question, k=1)
prompt = build_prompt_v3(question, chunks)

print("Answer:", generate(prompt))

Answer: (iii).


In [41]:
print(generate("What colour is the sky on a clear day?"))

blue


In [42]:
context = (
    "Coaching helps learners discover solutions through "
    "inquiry and reflection. Coaches empower learners "
    "to think independently and develop confidence."
)

prompt = f"""Answer the question based on the context.

Context: {context}

Question: Should coaches give learners the answers?

Answer:"""

print(generate(prompt))

Yes


In [43]:
for page_number, text in enumerate(pages_text, start=1):
    if "(iii)" in text:
        print(f"Found (iii) on page {page_number}")